In [3]:
import json
import pandas as pd
import math
from pathlib import Path
from difflib import SequenceMatcher
from typing import List, Dict, Tuple, Any
import json
import re
import os
from glob import glob
from collections import defaultdict, Counter
from statistics import mean
import ast
import numpy as np

# RQ1: Workflow execution limits

## Evaluation: Task success rate (1) history.is_successful() = True (2) post_verification check

In [4]:
verification_data_path = '../results/workflow_execution/with_verification_status.csv'
df_verification = pd.read_csv(verification_data_path)
total_entries = len(df_verification)
print(f"Total entries in verification data: {total_entries}")

# agent task success rate. Check 'is_successful'
count_successful = df_verification['is_successful'].sum()
success_rate = count_successful / total_entries if total_entries > 0 else 0
print(f"Agent Task Success Rate: \n\tSuccessful entries: {count_successful} | successful rate: {count_successful / total_entries:.2%} || blocked entries: {total_entries - count_successful} | Blocked rate: {(total_entries - count_successful) / total_entries:.2%}")


# post_verification block rate. Check 'blocked' and 'block_category'
blocked_count = df_verification['blocked'].sum()
unblocked_count = total_entries - blocked_count
block_rate = blocked_count / total_entries if total_entries > 0 else 0
print(f"Post-Verification Block Rate: \n\tSuccessful entries: {unblocked_count} | successful rate: {unblocked_count / total_entries:.2%} || blocked entries: {blocked_count} | Blocked rate: {blocked_count / total_entries:.2%}")

block_category_counts = Counter()
for category in df_verification['block_category']:
    if category != "no_blocking":
        block_category_counts[category] += 1

print("\nBlock Category Distribution:")
rows = []
for category, count in block_category_counts.items():
    percentage = count / blocked_count if blocked_count > 0 else 0
    print(f"\t{category}: {count} ({percentage:.2%})")
    rows.append({
        "category": category,
        "count": count,
        "percentage": f"{percentage:.2%}",
    })
df_block_category = pd.DataFrame(rows)


# overall task success rate, considering both agent task success and post-verification block. Only count entries with is_successful == True and blocked == False as successful.
overall_successful_count = df_verification[(df_verification['is_successful'] == True) & (df_verification['blocked'] == False)].shape[0]
overall_success_rate = overall_successful_count / total_entries if total_entries > 0 else 0
overall_unsuccessful_count = total_entries - overall_successful_count
print(f"\nOverall Task Success Rate (considering both agent success and post-verification block): \n\tSuccessful entries: {overall_successful_count} | overall successful rate: {overall_success_rate:.2%} || unsuccessful entries: {overall_unsuccessful_count} || overall unsuccessful rate: {(overall_unsuccessful_count) / total_entries:.2%}")


Total entries in verification data: 1112
Agent Task Success Rate: 
	Successful entries: 987 | successful rate: 88.76% || blocked entries: 125 | Blocked rate: 11.24%
Post-Verification Block Rate: 
	Successful entries: 895 | successful rate: 80.49% || blocked entries: 217 | Blocked rate: 19.51%

Block Category Distribution:
	automation_instability: 58 (26.73%)
	navigation_failure: 22 (10.14%)
	interaction_failure: 53 (24.42%)
	content_access_limitation: 17 (7.83%)
	security_barrier: 56 (25.81%)
	agent_reasoning_failure: 11 (5.07%)

Overall Task Success Rate (considering both agent success and post-verification block): 
	Successful entries: 877 | overall successful rate: 78.87% || unsuccessful entries: 235 || overall unsuccessful rate: 21.13%
